> **Experiment-source record.** This notebook is retained to document the code path used for the released run. Outputs and credentials have been removed. The canonical numerical reproduction starts from the persisted CSV/JSON records via `00_reproduce_manuscript_analysis.ipynb`.


# Axis 2 — new models, v2

Two protocol refinements are documented here:

**1. Repair the four models that produced no parseable label.** GLM-5.2, Kimi-K3, MiniMax-M3 and
Qwen3.5-397B all responded, none errored, and none emitted a label. They are reasoning models, and
a 512-token cap is consumed by deliberation before an answer is reached. Reporting them as
failures would be wrong: it would record a token-budget artefact as a capability result, and it
would drop the two genuinely frontier entrants that make the null result worth restating. Stage A
auto-negotiates a working configuration per model.

**2. Ten seeds, at a third of the cost per seed.** With `temperature=0` a three-vote majority
adds little beyond triple the spend, so the ten-seed run uses a single deterministic vote. That
puts ten seeds at roughly the price of the three already spent.

**3. Results are written after every model, and the archive cell can be run at any time.** The
previous session ended before its files were downloaded. Completed work is also skipped on
restart, so an interrupted run resumes instead of repeating.

Released campaign context:

| model | accuracy (3 seeds) | vs RF anchor | seeds won |
|---|---|---|---|
| RandomForest (anchor) | 0.808 | — | — |
| DeepSeek-V4-Pro | 0.526 | −0.282 | 0/3 |
| Nemotron-3-Ultra-550b-a55b | 0.229 | −0.579 | 0/3 |

Both lost on every seed by very large margins. Ten seeds will not change that verdict — it
tightens an interval around a 28-point and a 58-point deficit. The reason to run it is protocol
consistency with the reported panel, not doubt about the outcome. Stage A is the higher-value
spend.

In [ ]:
!pip -q install openai kagglehub pandas scikit-learn pyarrow 2>/dev/null

import json, os, re, time, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI

try:
    from google.colab import userdata
    NEBIUS_API_KEY = userdata.get("NEBIUS_API_KEY")
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try: os.environ.setdefault(k, userdata.get(k) or "")
        except Exception: pass
except Exception:
    NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")

assert NEBIUS_API_KEY, "Add NEBIUS_API_KEY to Colab secrets (key icon, left sidebar)."
client = OpenAI(base_url="https://api.studio.nebius.com/v1/", api_key=NEBIUS_API_KEY)

OUT = Path("axis2_new_models"); OUT.mkdir(exist_ok=True)

# Optional but recommended: persist to Drive so a session ending cannot lose the run.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/axis2_new_models"); OUT.mkdir(parents=True, exist_ok=True)
print("output dir:", OUT.resolve())

In [ ]:
TARGETS = {
    "GLM-5.2":                    ["glm-5.2", "glm5.2"],
    "DeepSeek-V4-Pro":            ["deepseek-v4-pro", "deepseekv4pro"],
    "Kimi-K3":                    ["kimi-k3", "kimik3"],
    "MiniMax-M3":                 ["minimax-m3", "minimaxm3"],
    "Qwen3.5-397B-A17B":          ["qwen3.5-397b", "397b-a17b"],
    "Nemotron-3-Ultra-550b-a55b": ["nemotron-3-ultra", "550b-a55b"],
}
available = sorted(m.id for m in client.models.list().data)
norm = lambda s: re.sub(r"[^a-z0-9.]", "", s.lower())
RESOLVED = {}
for name, pats in TARGETS.items():
    hit = next((m for m in available if any(norm(p) in norm(m) for p in pats)), None)
    if hit: RESOLVED[name] = hit
    print(f"  {'OK ' if hit else '-- '} {name:<28} {hit or 'not available'}")
assert RESOLVED

In [ ]:
import kagglehub
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

CIC_MAP = {"BENIGN":"BENIGN","Benign":"BENIGN","DDoS":"DDoS","DoS Hulk":"DoS",
           "DoS GoldenEye":"DoS","DoS slowloris":"DoS","DoS Slowhttptest":"DoS",
           "PortScan":"Port Scan","Portscan":"Port Scan","FTP-Patator":"Brute Force",
           "SSH-Patator":"Brute Force","Bot":"Bot","Infiltration":"Infiltration",
           "Heartbleed":"Heartbleed"}
def group(v):
    s = re.sub(r"[^\x00-\x7F]+", " ", str(v)).replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()
    if "web attack" in s.lower(): return "Web Attack"
    return CIC_MAP.get(s, CIC_MAP.get(str(v).strip(), s))

root = Path(kagglehub.dataset_download("dhoogla/cicids2017"))
files = sorted(list(root.rglob("*.parquet")) + list(root.rglob("*.csv")))
df = pd.concat([(pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f, low_memory=True))
                for f in files], ignore_index=True)
df.columns = [str(c).strip().replace(" ", "_") for c in df.columns]
lab = "ClassLabel" if "ClassLabel" in df.columns else "Label"
df["Label"] = df[lab].map(group)
num = df.select_dtypes(include=[np.number]).columns.tolist()
df = df[num + ["Label"]].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
print(f"{len(df):,} rows, {df.Label.nunique()} classes")

K_FEATURES, K_SHOTS, N_TEST_PER_CLASS, N_DEV_PER_CLASS = 12, 5, 30, 400

def three_way(d, seed):
    rng = np.random.default_rng(seed)
    dev, icl, test = [], [], []
    for _, part in d.groupby("Label", sort=True):
        part = part.iloc[rng.permutation(len(part))]
        n_dev = min(N_DEV_PER_CLASS, max(0, len(part) - K_SHOTS - N_TEST_PER_CLASS))
        rest = len(part) - n_dev
        n_icl = min(K_SHOTS, rest); n_test = min(N_TEST_PER_CLASS, max(0, rest - n_icl))
        dev.append(part.iloc[:n_dev]); icl.append(part.iloc[n_dev:n_dev+n_icl])
        test.append(part.iloc[n_dev+n_icl:n_dev+n_icl+n_test])
    cat = lambda ps: pd.concat([p for p in ps if len(p)], ignore_index=True)
    return cat(dev), cat(icl), cat(test)

def mi_features(d, k, seed):
    x = d.drop(columns=["Label"], errors="ignore").select_dtypes(include=[np.number])
    y = LabelEncoder().fit_transform(d["Label"].astype(str))
    return list(x.columns[np.argsort(mutual_info_classif(x, y, random_state=seed))[::-1][:k]])

LABELS = sorted(df.Label.astype(str).unique())
dev0, icl0, test0 = three_way(df, 0)
FEATURES = mi_features(dev0, K_FEATURES, 0)
print(f"dev={len(dev0)} icl={len(icl0)} test={len(test0)}")

## Prompting, parsing and per-model negotiation

`finish_reason` is the diagnostic that was missing. `length` means the budget ran out mid-answer,
which is a configuration problem; `stop` with no label means the model genuinely declined to
answer in the requested form. The previous run could not tell these apart.

Stage A tries progressively more accommodating configurations and keeps the first that works:

1. 512 tokens, plain — what the completed run used;
2. 4096 tokens — enough for a reasoning model to finish and then answer;
3. 4096 tokens with thinking disabled via `extra_body` — vendors expose different flags, so
   several are attempted;
4. 4096 tokens with an assistant prefill nudging the reply to start at the label.

Whichever succeeds is recorded per model and reported in the paper, because a model evaluated
with thinking disabled is not being evaluated under the same conditions as one that was not.

In [ ]:
def build_prompt(icl, row, features):
    shots = "\n".join(
        json.dumps({**{f: round(float(r[f]), 4) for f in features}, "label": str(r["Label"])})
        for _, r in icl.iterrows())
    system = ("You are a network intrusion detection classifier. "
              f"Choose exactly one label from this list: {', '.join(LABELS)}. "
              "Respond with the label only, on its own line, with no explanation.")
    return system, f"{shots}\n" + json.dumps({f: round(float(row[f]), 4) for f in features})

def parse_label(text):
    if not text: return None
    t = re.sub(r"<(think|thinking|reasoning)>.*?</\1>", " ", text, flags=re.S | re.I)
    t = re.sub(r"^.*</(think|thinking|reasoning)>", " ", t, flags=re.S | re.I)
    for line in [l.strip(" \t\"'`*.-:") for l in t.strip().splitlines()[::-1] if l.strip()]:
        for lab in sorted(LABELS, key=len, reverse=True):
            if line.lower() == lab.lower(): return lab
        for lab in sorted(LABELS, key=len, reverse=True):
            if lab.lower() in line.lower(): return lab
    return None

# Vendors disable reasoning differently; unsupported keys are rejected, so each is tried alone.
CONFIGS = [
    {"tag": "plain_512",        "max_tokens": 512,  "extra": None,  "prefill": False},
    {"tag": "plain_4096",       "max_tokens": 4096, "extra": None,  "prefill": False},
    {"tag": "nothink_template", "max_tokens": 4096, "prefill": False,
     "extra": {"chat_template_kwargs": {"thinking": False, "enable_thinking": False}}},
    {"tag": "nothink_effort",   "max_tokens": 4096, "prefill": False,
     "extra": {"reasoning_effort": "none"}},
    {"tag": "prefill_4096",     "max_tokens": 4096, "extra": None,  "prefill": True},
]

def call(model_id, system, user, cfg, temperature=0.0):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    if cfg.get("prefill"):
        msgs.append({"role": "assistant", "content": "Label:"})
    kw = dict(model=model_id, messages=msgs, max_tokens=cfg["max_tokens"],
              temperature=temperature, timeout=300)
    if cfg.get("extra"): kw["extra_body"] = cfg["extra"]
    r = client.chat.completions.create(**kw)
    ch = r.choices[0]
    u = getattr(r, "usage", None)
    return ((ch.message.content or ""), getattr(ch, "finish_reason", "?"),
            getattr(u, "prompt_tokens", 0) or 0, getattr(u, "completion_tokens", 0) or 0)

print("runner ready")

## Stage A — negotiate a working configuration

Three probe rows per model per configuration. Cheap, and it stops at the first configuration that
parses. The output table is a paper artefact: it documents exactly how each model was queried.

In [ ]:
RUN_STAGE_A = True
PROBE_ROWS = 3

MODEL_CONFIG, probe_rows = {}, []
if RUN_STAGE_A:
    for name, mid in RESOLVED.items():
        chosen = None
        for cfg in CONFIGS:
            hits, fr, preview = 0, "", ""
            try:
                for j in range(PROBE_ROWS):
                    system, user = build_prompt(icl0, test0.iloc[j], FEATURES)
                    txt, fr, ti, to = call(mid, system, user, cfg)
                    preview = preview or txt.strip().replace("\n", " ")[:110]
                    hits += int(parse_label(txt) is not None)
                err = ""
            except Exception as e:
                err = f"{type(e).__name__}: {str(e).splitlines()[0][:90]}"
            probe_rows.append({"model": name, "config": cfg["tag"], "parsed": hits,
                               "of": PROBE_ROWS, "finish_reason": fr,
                               "preview": preview, "error": err})
            status = f"{hits}/{PROBE_ROWS}" if not err else err[:60]
            print(f"  {name:<28} {cfg['tag']:<18} {status:<22} finish={fr}")
            if not err and hits == PROBE_ROWS:
                chosen = cfg; break
        if chosen:
            MODEL_CONFIG[name] = chosen
            print(f"  -> {name}: using {chosen['tag']}\n")
        else:
            print(f"  -> {name}: no configuration produced a parseable label; excluded\n")
    probe = pd.DataFrame(probe_rows); probe.to_csv(OUT / "stage_a_config_probe.csv", index=False)
    json.dump({k: v["tag"] for k, v in MODEL_CONFIG.items()},
              open(OUT / "stage_a_chosen_config.json", "w"), indent=2)

print(f"usable: {list(MODEL_CONFIG)}")
print(f"excluded: {[m for m in RESOLVED if m not in MODEL_CONFIG]}")
print("\nAn excluded model must be reported as a prompt-format incompatibility, with its")
print("finish_reason and a raw preview, not as a low-accuracy result.")

## Stage B — ten-seed benchmark

`temperature=0` with a single vote. Deterministic decoding makes a three-way majority nearly
redundant, and it removes two thirds of the cost and wall time.

Results are appended to `results.csv` after every model, and completed `(seed, model)` pairs are
skipped, so this survives a disconnect. Run the archive cell whenever convenient — it does not
need the run to have finished.

In [ ]:
# ============================================================
#  NEMOTRON CONFIGURATION RUN
#  Paste AFTER cell 7 (Stage A), BEFORE the Stage B benchmark cell.
#
#  FIRST: re-run the DATA cell (the one that builds `df` from
#  dhoogla/cicids2017). The previous version of this cell used `df` as a local
#  name for results.csv and clobbered your dataset, which is what produced
#  KeyError: 'Label'. Every name below is prefixed to make that impossible.
#
#  Why re-run Nemotron at all: Stage A chose plain_512 because the probe parsed
#  3/3 — but finish_reason was "length". The parser was scraping a label out of
#  truncated reasoning, not reading an answer. The 0.215 accuracy and the
#  degenerate prediction distribution (Bot 63, Infiltration 61 of 246) measure
#  the parser, not the model. My Stage A gate should have required
#  finish_reason == "stop".
# ============================================================
import pandas as pd
from pathlib import Path

_TARGET = "Nemotron-3-Ultra-550b-a55b"

# --- guard: the dataset must still be intact -------------------------------
assert "df" in globals() and "Label" in df.columns, (
    "`df` is missing or has no Label column. Re-run the data cell that builds df "
    "from dhoogla/cicids2017, then run this cell again.")
assert _TARGET in RESOLVED, f"{_TARGET} not in RESOLVED — re-run the model-resolution cell."
print(f"dataset OK: {len(df):,} rows, {df.Label.nunique()} classes")

# --- 1. force the configuration Stage A should have chosen ------------------
_forced = next(c for c in CONFIGS if c["tag"] == "plain_4096")
print(f"{_TARGET} -> {_forced['tag']} (max_tokens={_forced['max_tokens']})")

# --- 2. restrict Stage B to this model only ---------------------------------
MODEL_CONFIG = {_TARGET: _forced}
RESOLVED     = {_TARGET: RESOLVED[_TARGET]}
SEEDS        = list(range(10))

# --- 3. drop the stale rows, else the resume logic skips the model -----------
_RES = OUT/"results.csv"
if _RES.exists():
    _res = pd.read_csv(_RES)
    (_RES.with_suffix(".csv.bak")).write_bytes(_RES.read_bytes())
    _keep = _res[_res.model != _TARGET]
    print(f"removing {len(_res)-len(_keep)} stale rows for {_TARGET} "
          f"(backup: {_RES.with_suffix('.csv.bak').name})")
    _keep.to_csv(_RES, index=False)
    del _res, _keep

# --- 4. verify it terminates before spending the budget ---------------------
_system, _user = build_prompt(icl0, test0.iloc[0], FEATURES)
_txt, _fr, _ti, _to = call(RESOLVED[_TARGET], _system, _user, _forced)
_lab = parse_label(_txt)
print(f"\nprobe: finish_reason={_fr!r}  parsed={_lab!r}  completion_tokens={_to}")
print(f"preview: {_txt.strip()[:160]!r}")

if _fr != "stop":
    print("\nSTILL TRUNCATING. Do NOT run Stage B — the result would again measure the")
    print("parser rather than the model. Options, in order:")
    print("  a) try the nothink_template or nothink_effort config from CONFIGS;")
    print("  b) exclude the model and report it as format-limited, citing finish_reason")
    print("     and the degenerate prediction distribution. That is a legitimate result.")
elif _lab is None:
    print("\nTerminates cleanly but emits no parseable label. This is a format mismatch,")
    print("not a capability result — report it as such rather than as a low score.")
else:
    print("\nTerminates cleanly and parses. Run the Stage B cell; ~$27 for 10 seeds.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

SEEDS = list(range(10))
N_VOTE = 1
PRICES = {"GLM-5.2": (1.40, 4.40), "DeepSeek-V4-Pro": (1.75, 3.50), "Kimi-K3": (3.00, 15.00),
          "MiniMax-M3": (0.30, 1.20), "Qwen3.5-397B-A17B": (0.60, 3.60),
          "Nemotron-3-Ultra-550b-a55b": (1.00, 3.00)}

RESULTS_CSV = OUT / "results.csv"
done = set()
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev.seed, prev.model))
    print(f"resuming: {len(done)} (seed, model) pairs already complete")

def run_model(mid, cfg, icl, test, features, n_vote=N_VOTE):
    preds, raw, lat, errs, ti_s, to_s = [], [], [], 0, 0, 0
    for _, row in test.iterrows():
        system, user = build_prompt(icl, row, features)
        votes, texts = [], []
        for _ in range(n_vote):
            try:
                t0 = time.time()
                txt, fr, ti, to = call(mid, system, user, cfg,
                                       0.0 if n_vote == 1 else 0.7)
                lat.append(time.time() - t0); ti_s += ti; to_s += to; texts.append(txt)
                lab = parse_label(txt)
                if lab is None:
                    txt, fr, ti, to = call(mid, system, user + "\n\nAnswer with the label only.",
                                           cfg, 0.0)
                    ti_s += ti; to_s += to; texts.append(txt); lab = parse_label(txt)
                if lab: votes.append(lab)
            except Exception as e:
                errs += 1; texts.append(f"__error__ {type(e).__name__}: {str(e)[:110]}")
        preds.append(pd.Series(votes).mode().iat[0] if votes else None)
        raw.append(texts)
    return preds, raw, lat, errs, ti_s, to_s

for seed in SEEDS:
    dev, icl, test = three_way(df, seed)
    feats = mi_features(dev, K_FEATURES, seed)
    y_true = test["Label"].astype(str)
    print(f"\n=== seed {seed} | test={len(test)} ===")
    batch, raw_log = [], []

    if (seed, "RandomForest") not in done:
        rf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
        rf.fit(icl[feats].to_numpy(), icl["Label"].astype(str).to_numpy())
        p = rf.predict(test[feats].to_numpy())
        batch.append({"seed": seed, "model": "RandomForest", "kind": "anchor",
                      "config": "n/a", "accuracy": accuracy_score(y_true, p),
                      "macro_f1": f1_score(y_true, p, average="macro", zero_division=0),
                      "parse_rate": 1.0, "errors": 0, "usd": 0.0,
                      "latency_median_s": 0.0, "wall_min": 0.0})
        print(f"  RandomForest       acc={batch[-1]['accuracy']:.4f}")

    for name, cfg in MODEL_CONFIG.items():
        if (seed, name) in done:
            print(f"  {name:<28} already done, skipped"); continue
        t0 = time.time()
        preds, raw, lat, errs, ti, to = run_model(RESOLVED[name], cfg, icl, test, feats)
        yp = pd.Series(preds).fillna("__unparsed__").astype(str)
        pin, pout = PRICES.get(name, (0.0, 0.0))
        batch.append({"seed": seed, "model": name, "kind": "llm", "config": cfg["tag"],
                      "accuracy": accuracy_score(y_true, yp),
                      "macro_f1": f1_score(y_true, yp, average="macro", zero_division=0),
                      "parse_rate": float(np.mean([p is not None for p in preds])),
                      "errors": errs, "usd": (ti * pin + to * pout) / 1e6,
                      "latency_median_s": float(np.median(lat)) if lat else np.nan,
                      "wall_min": (time.time() - t0) / 60})
        raw_log.append({"seed": seed, "model": name, "config": cfg["tag"],
                        "features": feats, "y_true": y_true.tolist(),
                        "y_pred": preds, "raw": raw})
        r = batch[-1]
        print(f"  {name:<28} acc={r['accuracy']:.4f} f1={r['macro_f1']:.4f} "
              f"parse={r['parse_rate']:.2f} ${r['usd']:.2f} {r['wall_min']:.1f}min")

        # Persist immediately: the previous session ended before anything was saved.
        pd.DataFrame(batch).to_csv(RESULTS_CSV, mode="a", index=False,
                                   header=not RESULTS_CSV.exists())
        with (OUT / "raw_generations.jsonl").open("a") as fh:
            for g in raw_log: fh.write(json.dumps(g, default=str) + "\n")
        done.update((b["seed"], b["model"]) for b in batch)
        batch, raw_log = [], []

    if batch:
        pd.DataFrame(batch).to_csv(RESULTS_CSV, mode="a", index=False,
                                   header=not RESULTS_CSV.exists())
        done.update((b["seed"], b["model"]) for b in batch)

res = pd.read_csv(RESULTS_CSV).drop_duplicates(subset=["seed", "model"], keep="last")
print(f"\ntotal spend so far: ${res.usd.sum():.2f}")

In [ ]:
import itertools

summary = (res.groupby("model")
             .agg(n_seeds=("seed","nunique"), accuracy=("accuracy","mean"),
                  acc_sd=("accuracy","std"), macro_f1=("macro_f1","mean"),
                  parse_rate=("parse_rate","mean"), usd=("usd","sum"),
                  latency_median_s=("latency_median_s","mean"), errors=("errors","sum"))
             .sort_values("accuracy", ascending=False))
summary.to_csv(OUT / "summary.csv")
print(summary.to_string(float_format=lambda v: f"{v:9.4f}"))

def signflip(d):
    d = np.asarray(d, float); obs = abs(d.mean())
    return float(np.mean([abs(np.mean(d * np.array(s))) >= obs - 1e-15
                          for s in itertools.product((-1, 1), repeat=len(d))]))

anchor = res[res.model == "RandomForest"].set_index("seed").accuracy
rows = []
for name in [m for m in summary.index if m != "RandomForest"]:
    a = res[res.model == name].set_index("seed").accuracy
    common = sorted(set(a.index) & set(anchor.index))
    if not common: continue
    d = a.loc[common].to_numpy() - anchor.loc[common].to_numpy()
    rows.append({"model": name, "n_seeds": len(d), "mean_delta": d.mean(),
                 "min_delta": d.min(), "max_delta": d.max(),
                 "seeds_won": int((d > 0).sum()),
                 "exact_signflip_p": signflip(d) if len(d) <= 20 else np.nan,
                 "p_floor": 2.0 ** (-(len(d) - 1)),
                 "beats_anchor_every_seed": bool((d > 0).all())})
v = pd.DataFrame(rows); v.to_csv(OUT / "verdict.csv", index=False)
print("\npaired difference versus the Random Forest anchor:")
print(v.to_string(index=False, float_format=lambda x: f"{x: .4f}"))

if v.beats_anchor_every_seed.any():
    print("\nA candidate led on every seed. This bears directly on the Axis-2 conclusion;")
    print("run it across all four configurations before drafting.")
else:
    print("\nNo candidate surpassed the anchor. The null result stands, and now stands against")
    print("current frontier models. Report every screened model, including those excluded at")
    print("Stage A, with the configuration each required.")

## Archive — run this whenever, not only at the end

In [ ]:
import hashlib
from datetime import datetime, timezone

files = sorted(p for p in OUT.rglob("*") if p.is_file() and p.suffix != ".zip")
if not files:
    print("nothing to archive yet")
else:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    archive = Path(f"axis2_new_models_{stamp}.zip")
    manifest = {"generated_utc": datetime.now(timezone.utc).isoformat(), "files": []}
    for p in files:
        h = hashlib.sha256()
        with p.open("rb") as fh:
            for b in iter(lambda: fh.read(1 << 20), b""): h.update(b)
        manifest["files"].append({"path": p.name, "bytes": p.stat().st_size,
                                  "sha256": h.hexdigest()})
    (OUT / "ARCHIVE_MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files + [OUT / "ARCHIVE_MANIFEST.json"]:
            z.write(p, arcname=f"axis2_new_models/{p.name}")
    print(f"{archive}  ({archive.stat().st_size/1e6:.2f} MB, {len(files)+1} entries)")
    try:
        from google.colab import files as cf
        cf.download(str(archive))
    except Exception:
        print("absolute path:", archive.resolve())